# Transformer Models (BERT, GPT) for NLP
## AIAT 122 – Deep Learning

## Learning objectives
- Load BERT and get embeddings for a sentence (understanding).
- Load GPT-2 and generate a short text (generation).

**Where is this used in real life?** BERT powers search, classification, and NER; GPT powers chatbots and completion. **We use BERT for understanding** (bidirectional context) **and GPT for generation** (left-to-right); we use them instead of only RNNs because Transformers capture long-range dependencies in parallel and scale better.

**Prerequisites:** Basic PyTorch/transformers. Run `pip install transformers torch` if needed.

## Short theory
- **BERT:** Encoder-only; bidirectional; good for classification, NER, QA.
- **GPT:** Decoder-only; autoregressive; good for text generation.
- **Hugging Face:** Pre-trained models and tokenizers; we use them for embeddings and generation without training here.

**📌 Covers slide(s):** Optional — do after core notebooks 01–05 (no specific slide).


## Inputs & Outputs
**Inputs:** `transformers`, `torch`; pre-trained BERT and GPT-2 (downloaded on first run).  
**Dataset:** Real — pre-trained BERT and GPT-2 (Hugging Face; downloaded on first run).  
**Outputs:** BERT embedding shape for a sample sentence, and a short GPT-2 generated text. Run time: under ~5 min (download once).


In [1]:
%pip install transformers torch -q
from transformers import BertTokenizer, BertModel, GPT2Tokenizer, GPT2LMHeadModel
import torch
print("✅ Imports OK.")


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


✅ Imports OK.


### Step 1: BERT – get embeddings for a sentence

In [2]:
# We use BERT for understanding (bidirectional); good for classification/embeddings
bert_tok = BertTokenizer.from_pretrained("bert-base-uncased")
bert = BertModel.from_pretrained("bert-base-uncased")
sentence = "Machine learning is a subset of artificial intelligence."
inputs = bert_tok(sentence, return_tensors="pt")
with torch.no_grad():
    out = bert(**inputs)
# [CLS] or mean of last_hidden_state = sentence embedding
emb = out.last_hidden_state
print("BERT output shape (batch, seq_len, hidden):", emb.shape)
print("Sentence embedding (mean over tokens) shape:", emb.mean(dim=1).shape)

BERT output shape (batch, seq_len, hidden): torch.Size([1, 11, 768])
Sentence embedding (mean over tokens) shape: torch.Size([1, 768])


### Step 2: GPT-2 – generate short text from a prompt

In [3]:
# We use GPT for generation (decoder-only, left-to-right)
gpt_tok = GPT2Tokenizer.from_pretrained("gpt2")
gpt_tok.pad_token = gpt_tok.eos_token
gpt = GPT2LMHeadModel.from_pretrained("gpt2")
prompt = "The future of AI is"
inputs = gpt_tok(prompt, return_tensors="pt")
out = gpt.generate(inputs["input_ids"], max_new_tokens=25, do_sample=True, temperature=0.7, pad_token_id=gpt_tok.eos_token_id)
generated = gpt_tok.decode(out[0], skip_special_tokens=True)
print("Generated:", generated)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Generated: The future of AI is something we're all very excited about, so I'm excited to be part of it," said Rolfe.




## 🌍 Real-World Worked Example — Character-Level Text Generator

**Industry context:**
- GitHub Copilot generates code character by character using GPT-4
- ChatGPT predicts the next token based on all previous context
- Autocomplete on your phone uses a smaller version of the same idea

We build a **character-level language model** that learns to generate text token by token — the exact mechanism behind all LLMs.

In [ ]:
import torch, torch.nn as nn, torch.optim as optim
import numpy as np

torch.manual_seed(42)
# ── Training text ────────────────────────────────────────────────────────────
text = (
    "to be or not to be that is the question whether tis nobler in the mind "
    "to suffer the slings and arrows of outrageous fortune or to take arms against "
    "a sea of troubles and by opposing end them to die to sleep no more and by "
    "a sleep to say we end the heartache and the thousand natural shocks that "
    "flesh is heir to tis a consummation devoutly to be wished to die to sleep"
)

chars  = sorted(set(text))
c2i    = {c:i for i,c in enumerate(chars)}
i2c    = {i:c for c,i in c2i.items()}
VOCAB  = len(chars)
enc    = [c2i[c] for c in text]

SEQ_LEN = 20
X_list, y_list = [], []
for i in range(len(enc)-SEQ_LEN-1):
    X_list.append(enc[i:i+SEQ_LEN])
    y_list.append(enc[i+SEQ_LEN])
X_t = torch.tensor(X_list, dtype=torch.long)
y_t = torch.tensor(y_list, dtype=torch.long)

# ── LSTM Language Model ───────────────────────────────────────────────────
class CharLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(VOCAB, 32)
        self.lstm  = nn.LSTM(32, 128, batch_first=True, num_layers=2)
        self.fc    = nn.Linear(128, VOCAB)
    def forward(self, x):
        out,_ = self.lstm(self.embed(x))
        return self.fc(out[:,-1,:])

model   = CharLM()
opt     = optim.Adam(model.parameters(), lr=3e-3)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(200):
    model.train()
    perm = torch.randperm(len(X_t))[:256]  # mini-batch
    loss = loss_fn(model(X_t[perm]), y_t[perm])
    opt.zero_grad(); loss.backward(); opt.step()
    if epoch % 50 == 0:
        print(f"Epoch {epoch} — loss: {loss.item():.3f}")

# ── Text Generation (Greedy / Temperature Sampling) ──────────────────────
def generate(seed_str, steps=80, temperature=0.8):
    model.eval()
    chars_out = list(seed_str)
    ctx = [c2i.get(c, 0) for c in seed_str[-SEQ_LEN:]]
    for _ in range(steps):
        inp = torch.tensor([ctx[-SEQ_LEN:]]).long()
        with torch.no_grad():
            logits = model(inp)[0] / temperature
        probs = torch.softmax(logits, 0).numpy()
        next_c = np.random.choice(len(probs), p=probs)
        chars_out.append(i2c[next_c])
        ctx.append(next_c)
    return ''.join(chars_out)

print("\n── Generated Text ──────────────────────────────────────────────")
print(generate("to be or not", steps=100))
print("\nThis is exactly how ChatGPT generates text — one token at a time.")

## 🧩 Mini-exercise

**Try it:** Run BERT on a different sentence and print the embedding shape. Or generate a longer sequence with GPT-2 (e.g. max_length=50) and compare.

---

## Summary
**What you did:** Loaded BERT and got embedding shape for a sentence; loaded GPT-2 and generated a short continuation from a prompt.

**In real life you'd also:** Fine-tune BERT for classification, use GPT for longer generation with better decoding, and add task heads.

**The main idea:** BERT = understanding (bidirectional); GPT = generation (autoregressive); both are Transformer-based and pre-trained.

**Next:** `10_sentiment_analysis_translation_speech.ipynb` shows sentiment analysis with the pipeline API.

## 📚 References & Further Reading

**Papers:**
- Radford et al. (2019) — [GPT-2: Language Models are Unsupervised Multitask Learners](https://d4mucfpksywv.cloudfront.net/better-language-models/language_models_are_unsupervised_multitask_learners.pdf)
- Brown et al. (2020) — [GPT-3](https://arxiv.org/abs/2005.14165)

**Interactive:**
- [Karpathy's nanoGPT](https://github.com/karpathy/nanoGPT) — build GPT in 300 lines
- [The Unreasonable Effectiveness of RNNs](http://karpathy.github.io/2015/05/21/rnn-effectiveness/)

**State-of-the-Art:** GPT-4, Claude 3.5, Gemini 1.5 — all trained on trillions of tokens with transformer decoders.